# Week 5 — Monitoring, Drift Detection & Observability

### Watch a live model decay — and detect it before your users do

> **MLOps & Deployment: A From-Scratch Engineering Codex**
> Part of the applied-systems track. Like the rest of this curriculum, every
> abstraction here is *built before it is used*: we re-implement the core of each
> MLOps tool in plain Python/NumPy first, then map what we built onto the
> industry-standard equivalent so the magic is gone by the time you reach it.

---

## Learning objectives

By the end of this notebook you will be able to:

1. Distinguish data drift, concept drift, and label drift, and know which you can detect without labels.
2. Implement the Population Stability Index (PSI) and a KS-test drift detector from scratch.
3. Build a CUSUM change-point detector for streaming performance monitoring.
4. Design the three layers of ML observability (operational, data, model metrics).
5. Close the loop: turn a drift signal into an automated retraining trigger (back to Week 4).

## Prerequisites

- Weeks 0–4 (a deployed, CI-promoted model with request logging).
- Basic statistics: distributions, histograms, the idea of a hypothesis test.

## How to read this notebook

Each section has three layers:

- **🧠 Theory** — the systems problem and *why* it exists.
- **🔧 From scratch** — a minimal, dependency-light implementation you fully own.
- **🏭 In practice** — how the real tool (MLflow, Docker, K8s, etc.) does the same thing, and what it adds.

Run cells top-to-bottom. Everything is self-contained and works on a CPU-only laptop.

In [1]:
# --- Standard setup ---------------------------------------------------------
from __future__ import annotations
import os, sys, json, time, math, hashlib, random, pathlib, tempfile, shutil
from dataclasses import dataclass, field, asdict
from typing import Any, Callable

import numpy as np

np.random.seed(0)
random.seed(0)

# A scratch workspace that we clean up between runs of this notebook.
WORK = pathlib.Path(tempfile.mkdtemp(prefix="mlops_week_"))
print("Working directory for this notebook:", WORK)

# --- Content-addressable hashing (introduced & explained in Week 0) ---------
# Re-defined here so each notebook is fully self-contained and runnable alone.
def hash_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()[:12]

def hash_obj(obj) -> str:
    canonical = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return hash_bytes(canonical.encode())

def hash_array(a: np.ndarray) -> str:
    return hash_bytes(a.tobytes() + str(a.shape).encode() + str(a.dtype).encode())

def capture_environment() -> dict:
    import platform
    return {"python": sys.version.split()[0], "platform": platform.platform(),
            "numpy": np.__version__}

Working directory for this notebook: /tmp/mlops_week_478wngyd


## 1. 🧠 Why models decay

A deployed model's accuracy almost always degrades over time, even though the
weights never change. The world moves; the model doesn't. Three distinct failure
modes, and they are *not* interchangeable:

| Drift type | What changes | Formally | Detectable without labels? |
|------------|--------------|----------|----------------------------|
| **Data (covariate) drift** | input distribution | $P(X)$ changes, $P(Y\mid X)$ same | ✅ yes — watch the inputs |
| **Concept drift** | the input→output relationship | $P(Y\mid X)$ changes | ❌ hard — needs labels/outcomes |
| **Label drift** | output distribution | $P(Y)$ changes | partially — watch predictions |

The cruel asymmetry: **data drift is easy to detect but may be harmless; concept
drift is what actually hurts you but is hard to detect** because you usually don't
have ground-truth labels in real time. Good monitoring watches inputs *and*
predictions *and* (whenever labels eventually arrive) true performance.

We'll build detectors for each and a system that ties them to action.

## 2. 🔧 Setting up a reference and a stream

Monitoring always compares **live data** against a **reference** — usually the
training distribution. Let's establish a reference window, then simulate a
production stream that starts identical and slowly drifts.

In [2]:
ref_rng = np.random.RandomState(0)
# Reference: feature ~ N(0,1), captured at training time.
reference = ref_rng.normal(0, 1, 5000)

def production_batch(day: int, n=1000) -> np.ndarray:
    """Day 0 matches reference; the mean drifts +0.08/day and variance grows."""
    rng = np.random.RandomState(100 + day)
    mean_shift = 0.08 * day
    std_growth = 1.0 + 0.03 * day
    return rng.normal(mean_shift, std_growth, n)

print("Reference: mean=%.3f std=%.3f" % (reference.mean(), reference.std()))
for d in [0, 5, 15, 30]:
    b = production_batch(d)
    print(f"Day {d:>2}: mean=%.3f std=%.3f" % (b.mean(), b.std()))

Reference: mean=-0.015 std=0.985
Day  0: mean=-0.017 std=1.046
Day  5: mean=0.407 std=1.181
Day 15: mean=1.220 std=1.464
Day 30: mean=2.328 std=1.857


## 3. 🔧 From scratch: Population Stability Index (PSI)

PSI is the workhorse of industrial drift monitoring (it comes from credit
scoring). The idea: bin the reference, bin the live data with the *same* bins, and
measure how much probability mass moved between bins.

$$ \text{PSI} = \sum_{i} (p_i^{\text{live}} - p_i^{\text{ref}}) \cdot \ln\frac{p_i^{\text{live}}}{p_i^{\text{ref}}} $$

Rule of thumb: **PSI < 0.1** no significant drift; **0.1–0.25** moderate;
**> 0.25** major drift, investigate. Note this is symmetric and is actually the
*Jensen-style* sum of two KL divergences — but you don't need the theory to use it.

In [3]:
def psi(reference: np.ndarray, live: np.ndarray, n_bins=10, eps=1e-6) -> float:
    # Bin edges from the reference quantiles (equal-frequency bins are standard)
    quantiles = np.linspace(0, 100, n_bins + 1)
    edges = np.percentile(reference, quantiles)
    edges[0], edges[-1] = -np.inf, np.inf       # catch out-of-range live values
    ref_counts, _ = np.histogram(reference, bins=edges)
    live_counts, _ = np.histogram(live, bins=edges)
    ref_p = ref_counts / ref_counts.sum() + eps
    live_p = live_counts / live_counts.sum() + eps
    return float(np.sum((live_p - ref_p) * np.log(live_p / ref_p)))

print(f"{'day':>4} {'PSI':>8}  interpretation")
for d in [0, 2, 5, 10, 20, 30]:
    val = psi(reference, production_batch(d))
    level = "OK" if val < 0.1 else ("MODERATE" if val < 0.25 else "MAJOR DRIFT")
    print(f"{d:>4} {val:>8.4f}  {level}")

 day      PSI  interpretation
   0   0.0066  OK
   2   0.0342  OK
   5   0.2181  MODERATE
  10   0.5755  MAJOR DRIFT
  20   1.5089  MAJOR DRIFT
  30   2.0758  MAJOR DRIFT


PSI climbs monotonically as the simulated distribution slides away from the
reference. In production you'd compute this nightly per feature and alert when any
feature crosses 0.25. It's cheap, label-free, and interpretable — which is why
it's everywhere.

## 4. 🔧 From scratch: a two-sample KS test

PSI gives a magnitude; sometimes you want a *statistical* answer to "are these two
samples from the same distribution?" The Kolmogorov–Smirnov test uses the largest
gap between the two empirical CDFs:

$$ D = \sup_x | F_{\text{ref}}(x) - F_{\text{live}}(x) | $$

We'll implement the statistic directly (the heart of `scipy.stats.ks_2samp`).

In [4]:
def ks_2samp_statistic(a: np.ndarray, b: np.ndarray) -> float:
    """Largest vertical gap between the two empirical CDFs."""
    all_vals = np.sort(np.concatenate([a, b]))
    cdf_a = np.searchsorted(np.sort(a), all_vals, side="right") / len(a)
    cdf_b = np.searchsorted(np.sort(b), all_vals, side="right") / len(b)
    return float(np.max(np.abs(cdf_a - cdf_b)))

def ks_critical(n1, n2, alpha=0.05):
    """Approx critical value; D above this => reject 'same distribution'."""
    c_alpha = 1.358  # for alpha=0.05
    return c_alpha * math.sqrt((n1 + n2) / (n1 * n2))

for d in [0, 5, 15, 30]:
    live = production_batch(d)
    D = ks_2samp_statistic(reference, live)
    crit = ks_critical(len(reference), len(live))
    print(f"day {d:>2}: KS D={D:.4f}  critical={crit:.4f}  "
          f"{'DRIFT' if D > crit else 'no drift'}")

day  0: KS D=0.0226  critical=0.0470  no drift
day  5: KS D=0.1978  critical=0.0470  DRIFT
day 15: KS D=0.4064  critical=0.0470  DRIFT
day 30: KS D=0.6120  critical=0.0470  DRIFT


PSI and KS agree on the trend but answer different questions: PSI says *how much*
the population shifted (a business-interpretable magnitude), KS says *whether* the
shift is statistically significant. Mature monitoring uses both — KS to flag,
PSI to prioritise.

## 5. 🧠🔧 Concept drift: watching *performance*, not just inputs

Data drift watches `P(X)`. But the metric you actually care about is accuracy,
which depends on `P(Y|X)`. When labels arrive (often delayed — a loan defaults
months later), you monitor the **error stream** for a sudden change. The classic
tool is **CUSUM** (cumulative sum), a sequential change-point detector that fires
fast on a shift in the mean error.

CUSUM accumulates how far each observation is above an expected level; when the
running sum exceeds a threshold, it declares a change.

In [5]:
class CUSUM:
    """One-sided CUSUM change-point detector for a streaming error signal."""
    def __init__(self, target: float, slack: float, threshold: float):
        self.target = target        # expected (in-control) error level
        self.slack = slack          # allowable slack (k); ignore tiny deviations
        self.threshold = threshold  # decision threshold (h)
        self.s = 0.0
        self.history = []

    def update(self, x: float) -> bool:
        # accumulate deviation above (target + slack); floor at 0
        self.s = max(0.0, self.s + (x - self.target - self.slack))
        self.history.append(self.s)
        return self.s > self.threshold

# Simulate a per-request error stream: in-control mean error 0.10,
# then a concept shift at t=200 pushes mean error to 0.25.
rng = np.random.RandomState(7)
errors = np.concatenate([
    rng.normal(0.10, 0.03, 200),     # healthy
    rng.normal(0.25, 0.03, 200),     # after concept drift
])

cusum = CUSUM(target=0.10, slack=0.02, threshold=1.0)
alarm_at = None
for t, e in enumerate(errors):
    if cusum.update(e) and alarm_at is None:
        alarm_at = t
print(f"True change point : t=200")
print(f"CUSUM alarm fired : t={alarm_at}  (detection delay = {alarm_at-200} samples)")

True change point : t=200
CUSUM alarm fired : t=208  (detection delay = 8 samples)


In [6]:
# A compact ASCII view of the CUSUM statistic climbing after the change point
hist = np.array(cusum.history)
print("CUSUM statistic over time (each row = 40 samples, bar scaled):")
for start in range(0, len(hist), 40):
    chunk = hist[start:start+40]
    level = chunk.mean()
    bar = "█" * min(40, int(level * 8))
    flag = "  <-- threshold crossed" if chunk.max() > cusum.threshold else ""
    print(f"  t={start:>3}-{start+39:>3}: {bar}{flag}")

CUSUM statistic over time (each row = 40 samples, bar scaled):
  t=  0- 39: 
  t= 40- 79: 
  t= 80-119: 
  t=120-159: 
  t=160-199: 
  t=200-239: ████████████████████  <-- threshold crossed
  t=240-279: ████████████████████████████████████████  <-- threshold crossed
  t=280-319: ████████████████████████████████████████  <-- threshold crossed
  t=320-359: ████████████████████████████████████████  <-- threshold crossed
  t=360-399: ████████████████████████████████████████  <-- threshold crossed


CUSUM detects the shift within a handful of samples of the true change point —
far faster than waiting for a monthly accuracy report. The trade-off knobs
(`slack`, `threshold`) are the same latency-vs-false-alarm tension as Week 3's
batching: tighter thresholds detect faster but cry wolf more often.

## 6. 🧠 The three layers of ML observability

A complete monitoring system watches three layers, top to bottom. Skipping any
one leaves a blind spot:

1. **Operational metrics** (it's *up*) — latency p99, QPS, error rate, saturation.
   These are the Week 3 golden signals; standard SRE.
2. **Data metrics** (inputs are *sane*) — feature drift (PSI/KS), null rates,
   range violations, schema changes. Label-free, fast.
3. **Model metrics** (predictions are *good*) — prediction distribution drift,
   confidence calibration, and — when labels arrive — accuracy/CUSUM.

A subtle but critical point: a service can be **100% healthy operationally**
(layer 1 all green) while **silently making terrible predictions** (layer 3 on
fire). The dashboard that only shows latency is the most dangerous kind of
green — it earns trust it hasn't verified.

In [7]:
# A tiny monitoring 'controller' that combines all three layers into one status.
def monitor_day(day: int, reference: np.ndarray, recent_errors: np.ndarray | None = None):
    live = production_batch(day)
    report = {
        "operational": {"p99_ms": 42.0, "error_rate": 0.001, "status": "healthy"},  # layer 1
        "data": {                                                                    # layer 2
            "psi": round(psi(reference, live), 4),
            "ks_D": round(ks_2samp_statistic(reference, live), 4),
        },
        "model": {},                                                                 # layer 3
    }
    # Decision logic
    psi_val = report["data"]["psi"]
    report["data"]["drift_alert"] = psi_val > 0.25
    if recent_errors is not None:
        report["model"]["mean_error"] = round(float(recent_errors.mean()), 4)
    # Overall verdict
    report["action"] = "RETRAIN" if psi_val > 0.25 else ("WATCH" if psi_val > 0.1 else "OK")
    return report

for d in [0, 10, 30]:
    print(f"--- Day {d} monitoring report ---")
    print(json.dumps(monitor_day(d, reference), indent=2))

--- Day 0 monitoring report ---
{
  "operational": {
    "p99_ms": 42.0,
    "error_rate": 0.001,
    "status": "healthy"
  },
  "data": {
    "psi": 0.0066,
    "ks_D": 0.0226,
    "drift_alert": false
  },
  "model": {},
  "action": "OK"
}
--- Day 10 monitoring report ---
{
  "operational": {
    "p99_ms": 42.0,
    "error_rate": 0.001,
    "status": "healthy"
  },
  "data": {
    "psi": 0.5755,
    "ks_D": 0.3046,
    "drift_alert": true
  },
  "model": {},
  "action": "RETRAIN"
}
--- Day 30 monitoring report ---
{
  "operational": {
    "p99_ms": 42.0,
    "error_rate": 0.001,
    "status": "healthy"
  },
  "data": {
    "psi": 2.0758,
    "ks_D": 0.612,
    "drift_alert": true
  },
  "model": {},
  "action": "RETRAIN"
}


## 7. 🔧 Closing the loop: drift → retraining trigger

This is where the whole course connects. A drift alert shouldn't just page a
human — at maturity level 4 it **triggers the Week 4 CI pipeline automatically**,
which retrains on fresh data, runs the champion/challenger gate, and promotes only
if the new model genuinely wins. The loop closes itself.

In [8]:
def drift_triggered_retraining(reference, max_days=40, psi_threshold=0.25):
    """Simulate the closed loop: monitor daily; on major drift, 'trigger CI'."""
    events = []
    for day in range(max_days):
        val = psi(reference, production_batch(day))
        if val > psi_threshold:
            events.append((day, round(val, 4)))
            # In production this would call the Week 4 pipeline:
            #   trigger_github_actions_workflow('ml-ci', inputs={'reason': 'drift'})
            # which retrains, runs the gate, and promotes if the challenger wins.
            return {
                "drift_detected_day": day,
                "psi_at_trigger": round(val, 4),
                "action": "triggered ml-ci retraining workflow",
                "gate": "champion/challenger decides promotion (Week 4)",
            }
    return {"drift_detected_day": None, "action": "no retraining needed"}

print(json.dumps(drift_triggered_retraining(reference), indent=2))

{
  "drift_detected_day": 7,
  "psi_at_trigger": 0.2783,
  "action": "triggered ml-ci retraining workflow",
  "gate": "champion/challenger decides promotion (Week 4)"
}


## 8. 🏭 In practice

| You built | Real-world equivalent |
|-----------|----------------------|
| `psi`, `ks_2samp_statistic` | Evidently AI, NannyML, Fiddler, Arize drift reports |
| `CUSUM` | river's drift detectors (ADWIN, DDM, Page-Hinkley), NannyML performance estimation |
| three-layer `monitor_day` | Prometheus (ops) + a drift platform (data/model) + Grafana dashboards |
| `drift_triggered_retraining` | event-driven retraining: alert → Airflow/Kubeflow/GitHub Actions |

Real platforms add: confidence intervals on drift scores, multivariate drift
(not just per-feature), automatic root-cause attribution, and performance
*estimation* when labels are delayed (NannyML's signature trick). But every one of
them is built on the same primitives — compare to a reference, quantify the shift,
threshold, act — that you implemented here.

---

## 🎯 Exercises

Try these before moving on. They are deliberately open-ended — the goal is to
extend the machinery you just built, not to find a single right answer.

1. PSI uses equal-frequency bins from the reference. Show what happens to PSI if the live data has values entirely outside the reference range, and explain why the `-inf/+inf` edge clamp matters.
2. Implement multivariate drift: instead of per-feature PSI, train a classifier to distinguish reference from live samples. If it succeeds (AUC > 0.5 + margin), the distributions differ. Why does this catch drift that per-feature PSI misses?
3. CUSUM detects mean shifts. Add a two-sided version that also catches a drop in error (your model mysteriously got *better* — usually a data-pipeline bug). What real incident does 'error suddenly near zero' indicate?
4. Labels arrive 30 days late. Design a monitoring scheme that gives an *early* warning from inputs (layer 2) and a *confirmed* signal from labels (layer 3). How do you avoid both false alarms and slow detection?
5. Set the retraining PSI threshold too low and you retrain constantly; too high and you serve a stale model. Propose a cost model (retraining cost vs. cost of staleness) to choose the threshold rationally.

---

## ✅ Recap — Week 5

- Models decay because the world drifts while weights don't; data drift, concept drift, and label drift are distinct.
- PSI and the KS test detect input drift cheaply and without labels — but input drift may be harmless.
- Concept drift is what hurts and needs labels; CUSUM detects performance shifts fast on a streaming error signal.
- Observability has three layers — operational, data, model — and a service can be operationally green while silently failing.
- At full maturity, a drift alert triggers the Week 4 pipeline automatically, closing the train→serve→monitor→retrain loop.

### Coming up next

**Week 6 — LLM-Specific Deployment.** Everything so far applies to any model. Finally we tackle what makes LLMs uniquely hard to serve: the KV-cache, quantization, continuous batching, and the throughput/latency math of autoregressive generation.